In [ ]:
from os import path
import pandas as pd
from IPython.display import display
import numpy as np 
import sys 
import time 
import zarr
import glob
import os
import re
import matplotlib.pyplot as plt

pythonPackagePath = os.path.abspath(r'C:\Users\Lab admin\Desktop\LLSM-CME-ANALYSIS\Final\src')
sys.path.append(pythonPackagePath)

from extract_pixel_data import Extractor

Do not change the code in the cell below

In [ ]:
base_dir = r'C:\Users\Lab admin\Desktop\u-track3D\testTrackability'
input_file_directory = 'controlOS_analysis/'

# If a folder named 'datasets' doesn't exist in base_dir + input_file_directory, it will be created
if not os.path.exists(os.path.join(base_dir, input_file_directory, 'datasets')):
    os.makedirs(os.path.join(base_dir, input_file_directory, 'datasets'))

zarr_file_directory = input_file_directory + 'zarr_file/all_channels_data'
zarr_full_path = os.path.join(base_dir, zarr_file_directory)

input_directory = os.path.join(base_dir, input_file_directory) + 'tracks/Channel_1_tracking_result.pkl'

# List which channel was used for detection and tracking
channel_detected = 3

output_directory = 'datasets'
output_file_name = 'track_df_cleaned_final_full.pkl'
output_directory_full = os.path.join(base_dir, input_file_directory, output_directory, output_file_name)

In [ ]:
track_df = pd.read_pickle(input_directory)
# read the zarr file which contains the data of all three channels 
z2 = zarr.open(zarr_full_path, mode='r')

# Add columns sigma_z, sigma_y, sigma_x to track_df with values 3, 2, 2
track_df['sigma_z'] = 3
track_df['sigma_y'] = 2
track_df['sigma_x'] = 2

# Drop rows of track_df with NaN values in mu_z, mu_y, mu_x, and amplitude. Store the rows in track_df_cleaned, and the dropped rows in track_df_dropped
# Track df dropped should only drop those rows with NaN values in mu_z, mu_y, mu_x, and amplitude.
track_df_cleaned = track_df.dropna(subset=['mu_z', 'mu_y', 'mu_x', 'amplitude']).copy()
track_df_cleaned.loc[:, 'frame'] = track_df_cleaned['frame'] - 1
# Sort track_df_cleaned by frame
track_df_cleaned = track_df_cleaned.sort_values(by=['frame']).reset_index(drop=True)


In [ ]:
track_df_cleaned_copy = track_df_cleaned.copy()
track_df_cleaned_copy = track_df_cleaned_copy[track_df_cleaned_copy['track_id'] == 25817]

In [ ]:
track_df_cleaned.head()

The following parameters usually don't have to be adjusted, but you can adjust them based on the sizes of your spots


In [ ]:
# radii for sum intensity ("voxel sum") calculation, in [z,y,x]. a radius of 2 will measure 5 pixels across
radii_extractor = [3,2,2] 

# the radius (in pixels) of the shell for background subtraction
# used in "_voxel_sum_adjusted" calculation
background_radius_for_voxel_sum = [1,1,1] 

# number of CPUs used for parallel processing; this isn't currently used
n_cores = -1


Optionally set an offset value in XY between channel 2 and 3

In [ ]:
# If you see an offset between the cameras (channel 2 and 3), set that value here to correct for it.
# For example, if channel 2 shows up 4 pixels to the right of channel 2, set offset_x_ch2vs3 to 4

offset_x_ch2vs3 = 0
offset_y_ch2vs3 = 0

offset_ch2vs3 = [offset_y_ch2vs3, offset_x_ch2vs3]


# this line reverses the offset if you detected channel 2
if channel_detected == 2:
    offset_ch3vs2 = [-offset_ch2vs3[0], -offset_ch2vs3[1]]
else:
    offset_ch3vs2 = [0,0]
# if you don't know the offset, it will be estimated below

In [ ]:
extractor = Extractor(z2, dataframe = track_df_cleaned, radii=radii_extractor, frame_col_name = 'frame', 
                      radi_col_name = ['sigma_z', 'sigma_y', 'sigma_x'], n_jobs = n_cores)

# channels_list = [1,2,3]
channels_list = [3]

for channel in channels_list:

    offset = [0, 0]
    # need to verify that this offset works in the right direction
    if channel_detected == 2 and channel == 2:
        offset = offset_ch3vs2

    if channel_detected != 2 and channel == 2:
        offset = offset_ch2vs3

    col_names = ['mu_z', 'mu_y', 'mu_x']

    # # calculate mean, max, sum intensity around the track coordinates
    # peak_mean,peak_max,_,_,peak_loc,peak_sum = extractor.extract_pixels_data_fixed_bd(center_col_names = col_names, 
    #                                                     channel = channel, offset = offset)

    # calculate background subtracted sum intensity: "adjusted voxel sum"
    # voxel_sum_array, _, adj_voxel_sum, adj_voxel_sum_vol, vol_sig, vol_bg  = extractor.voxel_sum_fixed_background(center_col_names = ['mu_z', 'mu_y', 'mu_x'], channel = channel,                                                                            
        # background_radius=background_radius_for_voxel_sum, offset=offset)

    voxel_sum_array, _, _, adj_voxel_sum_vol, vol_sig, vol_bg  = extractor.voxel_sum_fixed_background(center_col_names = ['mu_z', 'mu_y', 'mu_x'], channel = channel,                                                                            
        background_radius=background_radius_for_voxel_sum, offset=offset)

    # track_df_cleaned[f'c{channel}_peak_mean']= peak_mean
    # track_df_cleaned[f'c{channel}_peak_max'] = peak_max
    # track_df_cleaned[f'c{channel}_voxel_sum'] = peak_sum
    # track_df[f'c{channel}_voxel_sum_adjusted'] = adj_voxel_sum
    track_df_cleaned[f'c{channel}_voxel_sum_adjusted'] = adj_voxel_sum_vol
    # track_df_cleaned[f'c{channel}_peak_x'] = [t[2] for t in peak_loc]
    # track_df_cleaned[f'c{channel}_peak_y'] = [t[1] for t in peak_loc]
    # track_df_cleaned[f'c{channel}_peak_z'] = [t[0] for t in peak_loc]


    
    print(f'Channel {channel} done')

# track_df.to_pickle(output_directory_full)
# print(f'Track intensities saved, you can continue to the next notebook.')
# track_df.head()

    


In [ ]:
track_df_cleaned[track_df_cleaned['track_id'] == 14942][['frame', 'mu_x', 'mu_y', 'mu_z', 'c1_voxel_sum_adjusted']]

In [ ]:
peak_mean,peak_max,_,_,peak_loc,peak_sum = extractor.extract_pixels_data_fixed_bd(center_col_names = col_names, 
                                                        channel = channel, offset = offset)

In [ ]:
extractor = Extractor(z2, dataframe = track_df_cleaned_copy, radii=radii_extractor, frame_col_name = 'frame', 
                      radi_col_name = ['sigma_z', 'sigma_y', 'sigma_x'], n_jobs = n_cores)

In [ ]:
# channel = 3
# offset = [0, 0]
current_channel = channel - 1 
frames = track_df_cleaned['frame'].nunique()
max_z = z2.shape[2]
max_y = z2.shape[3]
max_x = z2.shape[4]
voxel_sum_array = []
voxel_sum_array_max = []
pixel_values = []
pixel_values_max = []
adjusted_voxel_sum = []
adjusted_voxel_sum_by_volume = []
vol_sig = []
vol_bg = []

radius_z = 3
radius_y = 2
radius_x = 2

max_radius_z = 4
max_radius_y = 3
max_radius_x = 3

# volume_signal = (1+2*radius_z) * (1+2*radius_y) * (1+2*radius_x)
# volume_background = (1+2*max_radius_z) * (1+2*max_radius_y) * (1+2*max_radius_x)

# for frame in range(frames): 
#     current_df = track_df_cleaned[track_df_cleaned['frame'] == frame].reset_index()
#     current_image = z2[frame,current_channel,:,:,:]

frame = 26
current_df = track_df_cleaned[(track_df_cleaned['frame'] == frame)].reset_index()
current_image = z2[frame,current_channel,:,:,:]

    
for i in range(len(current_df)):

    z = current_df.loc[i,'mu_z']
    y = current_df.loc[i, 'mu_y'] - offset[0]
    x = current_df.loc[i, 'mu_x'] - offset[1]

    # y = max(0,current_df.loc[i, center_col_names[1]] - offset[0])
    # x = max(0,current_df.loc[i, center_col_names[2]] - offset[1])
    # # Ensure lower bounds for smaller patch 
    z_start = int(max(0, z - radius_z))
    y_start = int(max(0, y - radius_y))
    x_start = int(max(0, x - radius_x))

    # Ensure upper bounds for smaller patch 
    z_end = int(min(max_z, z + radius_z + 1))
    y_end = int(min(max_y, y + radius_y + 1))
    x_end = int(min(max_x, x + radius_x + 1))

    # Ensure lower bounds for larger patch 
    max_z_start = int(max(0, z - max_radius_z))
    max_y_start = int(max(0, y - max_radius_y))
    max_x_start = int(max(0, x - max_radius_x))

    # Ensure upper bounds for larger patch 
    max_z_end = int(min(max_z, z + max_radius_z + 1))
    max_y_end = int(min(max_y, y + max_radius_y + 1))
    max_x_end = int(min(max_x, x + max_radius_x + 1))

    # Extract relevant pixels for the smaller patch
    extracted_pixels = current_image[z_start:z_end, y_start:y_end, x_start:x_end]

    # Extract relevant pixels for the larger patch
    extracted_pixels_max = current_image[max_z_start:max_z_end, max_y_start:max_y_end, max_x_start:max_x_end]
    
    # Exclude pixels with value 0 before calculating mean
    non_zero_pixels = extracted_pixels[extracted_pixels != 0]
    volume_signal = non_zero_pixels.size

    # Exclude pixels with value 0 before calculating mean for the larger patch 
    non_zero_pixels_max = extracted_pixels_max[extracted_pixels_max != 0]
    volume_background = non_zero_pixels_max.size

    if non_zero_pixels.size > 0:
        # Calculate statistics
        voxel_sum = np.sum(non_zero_pixels)
        # print the voxel sum
        # print(f'Voxel sum for frame {frame}, track {current_df.loc[i, "track_id"]}, z={z}, y={y}, x={x}: {voxel_sum}')

        # Get coordinates of the maximum value
        voxel_sum_array.append(voxel_sum)
        pixel_values.append(non_zero_pixels)
    else:
        # If all pixels are 0, handle this case as needed
        voxel_sum_array.append(np.nan)  # Use NaN or any other suitable value

    if non_zero_pixels_max.size > 0:
        # Calculate statistics
        voxel_sum_max = np.sum(non_zero_pixels_max)

        # Get coordinates of the maximum value
        voxel_sum_array_max.append(voxel_sum_max)
        pixel_values_max.append(non_zero_pixels_max)
    else:
        # If all pixels are 0, handle this case as needed
        voxel_sum_array_max.append(np.nan)  # Use NaN or any other suitable value
    
    #adjusted voxel sum = small voxel sum - (large voxel sum - small voxel sum) * (AREA small / (AREA large - AREA small))
    area_small = non_zero_pixels.shape[0]
    area_large = non_zero_pixels_max.shape[0]
    # print(voxel_sum)

    # background_adjusted_voxel_sum = voxel_sum - ((voxel_sum_max - voxel_sum) * (area_small/ (area_large - area_small)))
    background_adjusted_voxel_sum = voxel_sum - ((float(voxel_sum_max) - float(voxel_sum)) * (float(area_small)/ (float(area_large) - float(area_small))))

    # background_adjusted_voxel_sum_by_volume = voxel_sum - ((voxel_sum_max - voxel_sum) * (volume_signal/ (volume_background - volume_signal)))
    background_adjusted_voxel_sum_by_volume = voxel_sum - ((float(voxel_sum_max) - float(voxel_sum)) * (float(volume_signal)/ (float(volume_background) - float(volume_signal))))    

    # if area_large == area_small:
    #     print('Warning: area_large and area_small are equal, which may lead to division by zero.')
    #     print(channel, frame, z, y, x)

    # if volume_background == volume_signal:
    #     print('Warning: volume_background and volume_signal are equal, which may lead to division by zero.')
    #     print(channel, frame, z, y, x, volume_background, volume_signal)

    # if (area_large - area_small) != 0:
    #     background_adjusted_voxel_sum = voxel_sum - ((float(voxel_sum_max) - float(voxel_sum)) * (float(area_small)/ (float(area_large) - float(area_small))))
    # else:
    #     # Have the background adjusted voxel sum equal NaN
    #     background_adjusted_voxel_sum = np.nan
    #     print('hi')

    # # background_adjusted_voxel_sum_by_volume = voxel_sum - ((voxel_sum_max - voxel_sum) * (volume_signal/ (volume_background - volume_signal)))
    # if (volume_background - volume_signal) != 0:
    #     background_adjusted_voxel_sum_by_volume = voxel_sum - ((float(voxel_sum_max) - float(voxel_sum)) * (float(volume_signal)/ (float(volume_background) - float(volume_signal))))
    # else:
    #     background_adjusted_voxel_sum_by_volume = np.nan
    #     print('hi')            
    
    adjusted_voxel_sum.append(background_adjusted_voxel_sum)
    adjusted_voxel_sum_by_volume.append(background_adjusted_voxel_sum_by_volume)
    vol_sig.append(volume_signal)
    vol_bg.append(volume_background)

In [ ]:
current_df[f'c{3}_voxel_sum_adjusted'] = adjusted_voxel_sum_by_volume

In [ ]:
current_df[current_df['track_id'] == 14942]
# current_df